# 🟢 cash — live feature tour

[cash](https://github.com/galgtonold/cash) caches your notebook **one statement at a time** and tracks how cells depend on each other, so re-running only recomputes what actually changed — nothing else.

This tour builds a small transaction-analytics pipeline and shows what cash does for you along the way. The whole thing runs in well under a minute.

**How to use this notebook**
1. **Runtime → Run all.** Every cell runs once and prints a cash **badge** summarizing what it did (`EXECUTED`, in ochre).
2. **Run all a _second_ time.** The expensive cells flip to green **`RESTORED`** — served instantly from cache. Nothing recomputes.
3. **Edit the highlighted cell near the bottom** (`TOP_N = …`) and Run all again — only that cell recomputes; the expensive pipeline above stays cached.
4. **Bonus:** *Runtime → Restart session*, then Run all — the cache survives a kernel restart, so the slow steps still restore.

In [ ]:
# Pre-release: install cash from GitHub.
# --force-reinstall --no-deps pulls the LATEST main even when a cached
# environment (a Binder image or Colab runtime) already has an older build
# under the same version number: a plain `pip install` would report "already
# satisfied" and silently keep the stale code. cash has no required deps, so
# --no-deps is safe and keeps this fast (numpy/pandas stay as-is).
# After the PyPI release this whole cell becomes:  %pip install -q "cash-lib[pandas]"
%pip install -q --force-reinstall --no-deps "git+https://github.com/galgtonold/cash.git"

import cash
import numpy as np
import pandas as pd

%cash_on

## 1 · An expensive pipeline that caches itself

First, a few million synthetic transactions. It's **seeded**, so the data is
identical every run — and cash caches the result, so building it is a one-time
cost.

In [ ]:
rng = np.random.default_rng(0)
N = 3_000_000

transactions = pd.DataFrame({
    "user_id":  rng.integers(1, 40_000, N),
    "category": rng.choice(["groceries", "travel", "tech", "health", "dining"], N),
    "amount":   rng.gamma(shape=2.0, scale=40.0, size=N).round(2),
})

print(f"{len(transactions):,} transactions  |  ${transactions.amount.sum():,.0f} total spend")
transactions.head()

Now the genuinely slow step — a grouped aggregation with a distinct-count over
millions of rows. This is exactly the kind of computation you don't want to
re-run every time you tweak something below it.

In [ ]:
category_stats = (
    transactions
    .groupby("category")
    .agg(total_spend=("amount", "sum"),
         avg_spend=("amount", "mean"),
         txns=("amount", "size"),
         unique_users=("user_id", "nunique"))
    .sort_values("total_spend", ascending=False)
)
category_stats

## 2 · Run it again — and it's free

Hit **Run all** a second time now. Watch the two cells above: their badges flip
from ochre `EXECUTED` to green **`RESTORED`**, and they finish instantly. cash
recognised that neither the code nor its inputs changed, so it handed back the
cached values instead of recomputing them.

## 3 · A reproducible simulation, cached

cash caches randomized computations too. Because the generator below is
**seeded**, the cached value is exactly reproducible — cash freezes it and says
so on the badge, no surprises.

In [ ]:
# Bootstrap a 95% confidence interval for the mean transaction amount.
sim_rng = np.random.default_rng(42)
amounts = transactions["amount"].to_numpy()

boot_means = np.empty(2000)
for i in range(2000):
    sample = sim_rng.choice(amounts, size=50_000, replace=True)
    boot_means[i] = sample.mean()

lo, hi = np.percentile(boot_means, [2.5, 97.5])
print(f"Mean transaction amount, 95% CI:  ${lo:.2f}  to  ${hi:.2f}")

## 4 · Smart invalidation — change one thing, recompute one thing

This is the payoff. The cell below depends on `category_stats` (cached above)
but is cheap. **Change `TOP_N` to another number and Run all again:** only this
cell recomputes — the multi-million-row pipeline above stays cached and instant.

In [ ]:
# 👇  EDIT THIS NUMBER and Run all again — only this cell recomputes.
TOP_N = 3

top = category_stats.head(TOP_N)

import matplotlib.pyplot as plt
ax = top["total_spend"][::-1].plot(kind="barh", color="#2e9e6b")
ax.set_title(f"Top {TOP_N} categories by total spend")
ax.set_xlabel("total spend ($)")
plt.tight_layout()
plt.show()

top

## 5 · Beyond notebooks — the `@cash.cache` decorator

Outside cells, wrap any function with `@cash.cache` and it caches by its
arguments and its own source code. The first call runs; an identical call
returns instantly.

In [ ]:
import time

@cash.cache
def spend_by_category(data, min_amount):
    time.sleep(1.0)                      # stand-in for a genuinely expensive step
    big = data[data["amount"] >= min_amount]
    return big.groupby("category")["amount"].mean().round(2)

print("first call (runs ~1s):")
print(spend_by_category(transactions, 50))
print("\nsecond call (instant, from cache):")
print(spend_by_category(transactions, 50))

## 6 · It survives a kernel restart

The cache lives on disk, not just in memory. Try **Runtime → Restart session**,
then **Run all**: the slow pipeline and the simulation restore from cache
instead of recomputing — a fresh kernel picks up right where you left off.

## What did cash save you?

Cache hits, misses, and the wall-clock time cash gave back this session:

In [ ]:
%cash_stats